# Task 2 – Violation features
**Input:** `food_clean.csv`   
**Output:** `joud_features.csv`, the violation columns, joined with Maram's columns at the end of the week.

In [27]:
import pandas as pd

We only need pandas here.

In [28]:
df = pd.read_csv('food_clean.csv')
df['Inspection Date'] = pd.to_datetime(df['Inspection Date'])
print(df.shape)

(126787, 20)


We read the clean file, the date becomes text again when saved to CSV, so we change it back to a real date.

In [29]:
# turning the violation text into a list of numbers, "38. INSECTS..." -> 38
def get_numbers(text):
    if pd.isnull(text):
        return []
    numbers = []
    for v in text.split(' | '):
        num = v.strip().strip('"').split('.')[0]
        if num.isdigit():
            numbers.append(int(num))
    return numbers

df['Viol_Numbers'] = df['Violations'].apply(get_numbers)
df[['Violations', 'Viol_Numbers']].head()

,Violations,Viol_Numbers
0,47. FOOD & NON-FOOD CONTACT SURFACES CLEANABLE...,"[47, 55]"
1,36. THERMOMETERS PROVIDED & ACCURATE - Comment...,"[36, 55]"
2,57. ALL FOOD EMPLOYEES HAVE FOOD HANDLER TRAIN...,[57]
3,NaN,[]
4,"3. MANAGEMENT, FOOD EMPLOYEE AND CONDITIONAL E...","[3, 5, 38, 48, 53, 58]"


**1.** Each violation in the text starts with its number and a dot, and violations are separated by ` | `. We keep only the numbers, and an empty text (no violations) gives an empty list.

In [30]:
# number of violations in each inspection
df['Num_Violations'] = df['Viol_Numbers'].apply(len)
df['Num_Violations'].describe()

count    126787.000000
mean          4.020562
std           3.566342
min           0.000000
25%           1.000000
50%           3.000000
75%           6.000000
max          34.000000
Name: Num_Violations, dtype: float64

**2.** The number of violations is the length of the list (mean =  4.020562, max = 34.000000 )

In [31]:
# one function for all topic columns
change_date = pd.Timestamp('2018-07-01')

def has_topic(row, old, new):
    if row['Inspection Date'] < change_date:
        codes = old
    else:
        codes = new
    for n in row['Viol_Numbers']:
        if n in codes:
            return 1
    return 0

**3.** The same topic has different numbers before and after 1 July 2018. This function checks the date of the inspection, then looks for the old numbers (before) or the new numbers (after), it returns 1 if the topic is found, 0 if not.
`apply(axis=1)` is new thing for us: it sends a whole row to the function, so the function can use two columns (date and numbers)

In [32]:
# old numbers = before 1 July 2018, new numbers = after (from violations_titles_before/after.csv)
df['has_rodents'] = df.apply(has_topic, axis=1, old=[13, 18], new=[38])
df['has_temperature'] = df.apply(has_topic, axis=1, old=[2, 3], new=[18, 19, 20, 21, 22, 33])
df['has_handwashing'] = df.apply(has_topic, axis=1, old=[6, 12], new=[8, 9, 10])
df['has_toxic'] = df.apply(has_topic, axis=1, old=[25], new=[28])
df['has_water_sewage'] = df.apply(has_topic, axis=1, old=[9, 10], new=[50, 52])
df['has_dirty_surfaces'] = df.apply(has_topic, axis=1, old=[33, 34, 35], new=[49, 55])

**Crosswalk (old → new), matched by the titles in the violation-list files:**

| Topic | Old numbers | New numbers |
|---|---|---|
| Rodents / insects | 13 (infestation), 18 (rodent proofing) | 38 |
| Temperature | 2, 3 | 18, 19, 20, 21, 22, 33 |
| Hand washing | 6 (hands, bare hand), 12 (sinks) | 8, 9, 10 |
| Toxic items | 25 | 28 |
| Water / sewage | 9 (water), 10 (sewage) | 50, 52 |
| Dirty surfaces | 33, 34, 35 | 49, 55 |

The two lists are not one-to-one, so this is an approximation.

We removed new #51 (plumbing / backflow) from water / sewage: in the old list plumbing was part of #38 (ventilation, plumbing), so #51 has no match in old #9 or #10.

In [33]:
# share of inspections with each topic, per period
topics = ['has_rodents', 'has_temperature', 'has_handwashing', 'has_toxic', 'has_water_sewage', 'has_dirty_surfaces']
df.groupby('Period')[topics].mean().round(3)

,has_rodents,has_temperature,has_handwashing,has_toxic,has_water_sewage,has_dirty_surfaces
Period,,,,,,
Grace (90 days),0.153,0.109,0.198,0.004,0.035,0.558
Late transition,0.155,0.077,0.237,0.004,0.033,0.553
Post,0.161,0.071,0.206,0.004,0.033,0.532
Pre,0.109,0.081,0.037,0.001,0.013,0.676


**4.** If a topic's share jumps a lot between Pre and Post, the matching may be wrong (or the rule really changed).

In [34]:
# which new number makes the jump? (share of Post inspections with each number)
after = df[df['Period'] == 'Post']
for n in [8, 9, 10, 50, 51, 52]:
    print(n, after['Viol_Numbers'].apply(lambda nums: n in nums).mean().round(3))

8 0.006
9 0.005
10 0.201
50 0.013
51 0.232
52 0.021


**Check of the jumps.** Hand washing goes from about 4% (Pre) to about 21% (Post), almost all of it comes from new #10 “Adequate handwashing sinks properly supplied and accessible” (20.1% of Post inspections), while #8 and #9 are under 1%, the new code checks sink supplies (soap, towels) much more strictly, so this is a real rule change, not a matching mistake, we keep the column, but its meaning is slightly different before and after 2018.

New #51 “Plumbing installed; proper backflow devices” appears in 23.2% of Post inspections. this is why water/sewage jumped before; after removing #51 it is close to the Pre level.

Temperature and toxic items stay about the same before and after, so their matching works well. Rodents rise a little (11% → 16%) and dirty surfaces fall (68% → 53%). Hand washing rises a lot because of the new #10. This supports H1: the mix of violations changes after the 2018 code.

In [35]:
cols = ['Inspection ID', 'Num_Violations'] + topics
df[cols].to_csv('joud_features.csv', index=False)
print(df[cols].shape)

(126787, 8)


We save only the ID and the new columns, at the end of the week this file is joined with Maram's file on `Inspection ID`.